# Agentic RAG System: Parts 1-8 (full, v2 fixed)
**Setup → Ingestion → Embeddings → ChromaDB → Corrective Retrieval → Answer with Sources → Tools → Planner + Agent → Standard vs Agentic → Gradio app**

Run the cells **top to bottom** in one Colab session. Parts 1-4 are the Corrective RAG engine; Parts 5-8 turn it into an **Agentic RAG** system.

## 1. Setup

In [1]:
import zipfile, shutil
from pathlib import Path

DATA_DIR = Path("/content/data")
SUPPORTED = {".pdf", ".docx", ".pptx", ".txt", ".csv"}

def count_files(d):
    return sum(1 for f in Path(d).rglob("*") if f.suffix.lower() in SUPPORTED)

if not DATA_DIR.exists() or count_files(DATA_DIR) == 0:
    candidates = [Path("/content/data.zip")] + sorted(Path("/content").glob("*.zip"))
    if Path("/content/drive/MyDrive").exists():
        candidates += sorted(Path("/content/drive/MyDrive").glob("*.zip"))
    zips = [z for z in candidates if z.exists()]
    if zips:
        print("Extracting:", zips[0])
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("/content/_unzipped")
        root = Path("/content/_unzipped")
        subs = [p for p in root.iterdir() if p.is_dir()]
        if len(subs) == 1 and any(p.is_dir() for p in subs[0].iterdir()):
            root = subs[0]
        DATA_DIR.mkdir(exist_ok=True)
        for p in root.iterdir():
            if p.is_dir() and p.name != "__MACOSX":
                shutil.move(str(p), str(DATA_DIR / p.name))

if not DATA_DIR.exists() or count_files(DATA_DIR) == 0:
    DATA_DIR.mkdir(exist_ok=True)
    for name in ["course1", "course2", "course3", "M269", "TM112", "TM275"]:
        if Path("/content", name).exists():
            shutil.move(str(Path("/content", name)), str(DATA_DIR / name))

RENAME = {"course1": "TM112", "course2": "TM275", "course3": "M269"}
for old, new in RENAME.items():
    if (DATA_DIR / old).exists() and not (DATA_DIR / new).exists():
        (DATA_DIR / old).rename(DATA_DIR / new)

if count_files(DATA_DIR) == 0:
    raise FileNotFoundError("No course files found. Upload data.zip (with the course folders) to /content, then re-run.")

for course in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    files = [f.name for f in course.iterdir() if f.suffix.lower() in SUPPORTED]
    print(course.name, "->", len(files), "files")
    for f in files: print("   ", f)

M269 -> 5 files
    M269_lec2_slides.csv
    M269_lec2.pptx
    m269.docx
    M269_lec2_notes.txt
    100-M269_MTA_REVISION.pdf
TM112 -> 5 files
    TM112 Meeting12-Cryptography_notes.txt
    Moc TM112 answerkey.docx
    TM112 Meeting12-Cryptography.pptx
    TM112 Meeting12-Cryptography_slides.csv
    Meeting 6 جاهز للطباعة.pdf
TM275 -> 5 files
    TM275 - Meeting 6# AWS Cloud (2).pptx
    TM275 - Meeting 6# AWS Cloud (2)_slides.csv
    TM275 - Meeting 3# Distributed Systems (Theoretical + Docker).pdf
    TM275 - Meeting 6# AWS Cloud (2)_notes.txt
    MTA Focus exam (1).docx


In [2]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/datasets
!cp -n "/content/per_class_test.csv" /content/datasets/ 2>/dev/null || true

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip -q install pypdf python-docx python-pptx chromadb sentence-transformers langchain-text-splitters pandas gradio groq

## 2. Unzip + course names
Unzips `data.zip` (if needed) and renames `course1/2/3` to the real course codes. Safe to re-run.

In [4]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/data.zip")
DATA_DIR = Path("/content/data")

if ZIP_PATH.exists() and not DATA_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content/")

# if the zip had the course folders at its root, they are in /content/ directly
if not DATA_DIR.exists():
    DATA_DIR.mkdir()
    for name in ["course1", "course2", "course3", "M269", "TM112", "TM275"]:
        if Path("/content", name).exists():
            Path("/content", name).rename(DATA_DIR / name)

RENAME = {"course1": "TM112", "course2": "TM275", "course3": "M269"}
for old, new in RENAME.items():
    if (DATA_DIR / old).exists() and not (DATA_DIR / new).exists():
        (DATA_DIR / old).rename(DATA_DIR / new)

SUPPORTED = {".pdf", ".docx", ".pptx", ".txt", ".csv"}
for course in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    files = [f.name for f in course.iterdir() if f.suffix.lower() in SUPPORTED]
    print(course.name, "->", len(files), "files")
    for f in files: print("   ", f)

M269 -> 5 files
    M269_lec2_slides.csv
    M269_lec2.pptx
    m269.docx
    M269_lec2_notes.txt
    100-M269_MTA_REVISION.pdf
TM112 -> 5 files
    TM112 Meeting12-Cryptography_notes.txt
    Moc TM112 answerkey.docx
    TM112 Meeting12-Cryptography.pptx
    TM112 Meeting12-Cryptography_slides.csv
    Meeting 6 جاهز للطباعة.pdf
TM275 -> 5 files
    TM275 - Meeting 6# AWS Cloud (2).pptx
    TM275 - Meeting 6# AWS Cloud (2)_slides.csv
    TM275 - Meeting 3# Distributed Systems (Theoretical + Docker).pdf
    TM275 - Meeting 6# AWS Cloud (2)_notes.txt
    MTA Focus exam (1).docx


## 3. Extraction
Each loader returns `(text, location)` so every chunk can be cited with its **file + page/slide**.
`_notes.txt` and `_slides.csv` files are skipped because they are just exports of the `.pptx`.

In [5]:
import pandas as pd
from pypdf import PdfReader
from docx import Document
from pptx import Presentation

def load_pdf(p):
    out = []
    for i, page in enumerate(PdfReader(str(p)).pages, 1):
        t = page.extract_text() or ""
        if t.strip():
            out.append((t, f"page {i}"))
    return out

def load_docx(p):
    d = Document(str(p))
    parts = [para.text for para in d.paragraphs if para.text.strip()]
    for tbl in d.tables:
        for row in tbl.rows:
            parts.append(" | ".join(c.text.strip() for c in row.cells))
    return [("\n".join(parts), "document")]

def load_pptx(p):
    out = []
    for i, slide in enumerate(Presentation(str(p)).slides, 1):
        texts = []
        for sh in slide.shapes:
            if sh.has_text_frame:
                texts.append(sh.text_frame.text)
            if getattr(sh, "has_table", False) and sh.has_table:
                for r in sh.table.rows:
                    texts.append(" | ".join(c.text for c in r.cells))
        if slide.has_notes_slide:
            texts.append(slide.notes_slide.notes_text_frame.text)
        t = "\n".join(x for x in texts if x.strip())
        if t.strip():
            out.append((t, f"slide {i}"))
    return out

def load_txt(p):
    return [(Path(p).read_text(encoding="utf-8", errors="ignore"), "text file")]

def load_csv(p):
    df = pd.read_csv(p, encoding="utf-8", on_bad_lines="skip").fillna("")
    rows = ["; ".join(f"{c}: {v}" for c, v in r.items() if str(v).strip()) for _, r in df.iterrows()]
    return [("\n".join(rows), "csv")]

LOADERS = {".pdf": load_pdf, ".docx": load_docx, ".pptx": load_pptx, ".txt": load_txt, ".csv": load_csv}

raw_docs = []   # {"text", "course", "source", "loc"}
for course in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    for f in sorted(course.iterdir()):
        if f.stem.endswith(("_notes", "_slides")):   # derived from the .pptx, skip duplicates
            continue
        if f.suffix.lower() == ".csv" and f.stat().st_size > 2_000_000:   # big table -> analyze_data tool, not RAG
            print("Skipping large CSV (used by the analyze_data tool):", f.name)
            continue
        loader = LOADERS.get(f.suffix.lower())
        if not loader:
            continue
        try:
            for text, loc in loader(f):
                raw_docs.append({"text": text, "course": course.name, "source": f.name, "loc": loc})
        except Exception as e:
            print("FAILED:", f.name, "->", e)

print("Extracted units:", len(raw_docs))
print(pd.DataFrame(raw_docs).groupby(["course", "source"]).size())

Extracted units: 253
course  source                                                           
M269    100-M269_MTA_REVISION.pdf                                            14
        M269_lec2.pptx                                                       46
        m269.docx                                                             1
TM112   Meeting 6 جاهز للطباعة.pdf                                            9
        Moc TM112 answerkey.docx                                              1
        TM112 Meeting12-Cryptography.pptx                                    40
TM275   MTA Focus exam (1).docx                                               1
        TM275 - Meeting 3# Distributed Systems (Theoretical + Docker).pdf    42
        TM275 - Meeting 6# AWS Cloud (2).pptx                                99
dtype: int64


## 4. Cleaning

In [6]:
import re

def fix_doubled_chars(t: str) -> str:
    """Some PDFs render bold text twice (e.g. 'AAllggoorriitthhmmss'). Collapse words made only of doubled letters."""
    def fix(m):
        w = m.group(0)
        if (len(w) >= 4 and len(w) % 2 == 0 and any(ch.isalpha() for ch in w)
                and all(w[i] == w[i + 1] for i in range(0, len(w), 2))):
            return w[::2]
        return w
    return re.sub(r"\w+", fix, t)

def clean_text(t: str) -> str:
    t = fix_doubled_chars(t)
    t = t.replace("\x00", " ")
    t = re.sub(r"(?im)^\s*(page\s*)?\d{1,3}\s*$", "", t)   # lone page numbers
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

for d in raw_docs:
    d["text"] = clean_text(d["text"])
raw_docs = [d for d in raw_docs if len(d["text"]) > 30]
print("After cleaning:", len(raw_docs), "units")

After cleaning: 243 units


## 5. Chunking
~900 characters, 150 overlap, split at paragraph/sentence boundaries (Arabic punctuation included).

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import hashlib

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900, chunk_overlap=150,
    separators=["\n\n", "\n", ". ", "؟ ", "، ", " ", ""],
)

chunks, seen = [], set()
for d in raw_docs:
    for piece in splitter.split_text(d["text"]):
        piece = piece.strip()
        if len(piece) < 40:
            continue
        h = hashlib.md5((d["course"] + re.sub(r"\s+", " ", piece.lower())).encode()).hexdigest()
        if h in seen:
            continue
        seen.add(h)
        chunks.append({"id": h, "text": piece, "course": d["course"], "source": d["source"], "loc": d["loc"]})

print("Total chunks:", len(chunks))
print(pd.DataFrame(chunks).groupby("course").size())
print("\nSample chunk:\n", chunks[0])

Total chunks: 296
course
M269      79
TM112     89
TM275    128
dtype: int64

Sample chunk:
 {'id': '8b7b9943f68892f4ccabd9d26a4d86a2', 'text': '1 M269 – Algorithm and Data structure – MTA Revision \n \n \n \n \nAlgorithms && Data Structure Using python \nM269 MTA sheet', 'course': 'M269', 'source': '100-M269_MTA_REVISION.pdf', 'loc': 'page 1'}


## 6. Embeddings + Vector DB
`intfloat/multilingual-e5-small` (Arabic + English, uses `passage:` / `query:` prefixes) and ChromaDB with cosine similarity.

In [8]:
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer("intfloat/multilingual-e5-small")

client = chromadb.PersistentClient(path="chroma_db")
try:
    client.delete_collection("course_docs")
except Exception:
    pass
collection = client.create_collection("course_docs", metadata={"hnsw:space": "cosine"})

BATCH = 128
for i in range(0, len(chunks), BATCH):
    batch = chunks[i:i + BATCH]
    embs = embedder.encode(["passage: " + c["text"] for c in batch],
                           normalize_embeddings=True, show_progress_bar=False).tolist()
    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        embeddings=embs,
        metadatas=[{"course": c["course"], "source": c["source"], "loc": c["loc"]} for c in batch],
    )
print("Stored in Chroma:", collection.count())

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Stored in Chroma: 296


## 7. Retrieval test

In [9]:
def retrieve(query: str, k: int = 5, course: str | None = None):
    q = embedder.encode(["query: " + query], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q, n_results=k,
                           where={"course": course} if course else None)
    return [
        {"text": t, "meta": m, "score": 1 - d}     # cosine similarity
        for t, m, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]

for query in ["What is the internet?", "What is symmetric encryption?"]:
    print("Q:", query)
    for r in retrieve(query, k=3):
        print(f"  {r['score']:.3f} | {r['meta']['course']} | {r['meta']['source']} | {r['meta']['loc']}")
        print("   ", r["text"][:200].replace("\n", " "))
    print()

Q: What is the internet?
  0.875 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 6
    • Lots of different networks of different sizes and shapes, all linked together to form the internet.  • When a network (LAN) joins to other networks (i.e., becomes part of the internet) – it has to c
  0.865 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 1
    Ahmed  Elbhbaty   Meeting #6 Network technologies | The Internet    ➢ Network of networks    • The term internet derives from that the internet is itself made up of many other joined-together networks
  0.864 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 4
    → Internet (IP) addresses: these have provided a universal addressing scheme.  → Protocols: these have allowed many new services to be added to existing architecture.  • It is the technology that unde

Q: What is symmetric encryption?
  0.887 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 32
    It is well-suited to encrypting any amount of data. Unlike asymmetric encryption, symmetric encry

# Part 2: Evaluate the retrieved chunks

For every question: retrieve top-k, an LLM grades each chunk (`relevant` / `ambiguous` / `irrelevant`), then the system decides:
- `correct`: at least one relevant chunk, keep only the relevant ones
- `ambiguous`: no clearly relevant chunk but some partial ones, so rewrite the query (Part 3)
- `incorrect`: nothing useful, so rewrite, and if still weak answer "not found"

### LLM setup (Groq)
Add `GROQ_API_KEY` in Colab Secrets, or paste it when asked. All LLM calls go through `call_llm`, so switching to Gemini later means editing one function.

Note: Groq retired `llama-3.3-70b-versatile` on Aug 16, 2026, so the notebook auto-picks `openai/gpt-oss-120b` (Groq's recommended replacement).

In [10]:
import os, json, time
from getpass import getpass
from groq import Groq

def _get_groq_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")          # Colab Secrets (name must be exactly GROQ_API_KEY, Notebook access ON)
    except Exception:
        key = None
    if not key:
        key = getpass("Groq API key (starts with gsk_): ")
    return key.strip().strip('"').strip("'")         # remove accidental spaces / quotes

os.environ["GROQ_API_KEY"] = _get_groq_key()
if not os.environ["GROQ_API_KEY"].startswith("gsk_"):
    raise ValueError("That does not look like a Groq key (it must start with gsk_). Create one at https://console.groq.com/keys")
llm_client = Groq()

# Groq retired llama-3.3-70b-versatile (Aug 2026). Pick the first preferred model your key can access.
PREFERRED = ["openai/gpt-oss-120b", "qwen/qwen3.8-27b", "openai/gpt-oss-20b"]
try:
    available = {m.id for m in llm_client.models.list().data}
except Exception as e:
    if "401" in str(e) or "Invalid API Key" in str(e):
        raise SystemExit("Groq rejected the key (401 Invalid API Key). Create a NEW key at console.groq.com/keys, update the GROQ_API_KEY secret, then Runtime > Restart session and run again.")
    raise
MODEL = next((m for m in PREFERRED if m in available), None)
if MODEL is None:
    print("None of the preferred models found. Available models:")
    for m in sorted(available): print("  ", m)
    raise SystemExit("Set MODEL manually to one of the text models above.")
print("Using model:", MODEL)

def call_llm(system: str, user: str, json_mode: bool = False,
             temperature: float = 0.0, max_tokens: int = 2000, retries: int = 3) -> str:
    kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
    if "gpt-oss" in MODEL:
        kwargs["reasoning_effort"] = "low"     # reasoning model: keep it fast, leave room for the answer
    for attempt in range(retries):
        try:
            r = llm_client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                temperature=temperature, max_tokens=max_tokens, **kwargs)
            return r.choices[0].message.content
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"LLM error ({e.__class__.__name__}: {str(e)[:120]}), retrying...")
            time.sleep(2 * (attempt + 1))

print(call_llm("You are helpful.", "Say OK."))

Groq API key (starts with gsk_): ··········
Using model: openai/gpt-oss-120b
OK.


### Chunk grader
One LLM call grades all k chunks together (saves API calls).

In [11]:
GRADER_SYSTEM = """You are a relevance grader for a document question-answering system.
Given a question and numbered text chunks, judge whether each chunk contains information that helps answer the question.
Verdicts:
- "relevant": the chunk contains information that directly helps answer the question. A formal one-line definition is NOT required: explanations, properties, purposes, examples or key facts that together answer the question count as relevant.
- "ambiguous": the chunk is about a related topic but only touches the question tangentially, or the key information is missing.
- "irrelevant": a different topic, or only titles / table of contents / headings with no explanatory content.
Do not rely on keyword overlap alone; judge by meaning.
- For "What is X?" / "Explain X" / "ما هو X" questions, a chunk that explains how X works, its properties, purpose, advantages or usage IS relevant (score 0.7 or more) even if it never gives a one-line definition.
- Never penalize a chunk because some information is missing from it: other chunks may supply it. Use "ambiguous" only when the chunk is about a different but related topic.
The question and chunks may be in Arabic or English.
Scores: relevant >= 0.7, ambiguous 0.3-0.6, irrelevant < 0.3.
Reply ONLY with JSON."""

REL_THRESHOLD = 0.6   # score needed to count a chunk as relevant
RESCUE_GRADE = 0.5    # an 'ambiguous' chunk with at least this grade ...
RESCUE_SIM   = 0.86   # ... AND at least this embedding similarity is kept as relevant (the grader can be too strict)

def grade_chunks(question: str, chunks: list[dict]) -> list[dict]:
    numbered = "\n\n".join(f"[{i}] {c['text'][:800]}" for i, c in enumerate(chunks))
    user = (f"Question: {question}\n\nChunks:\n{numbered}\n\n"
            'Return JSON: {"grades": [{"index": 0, "verdict": "relevant|ambiguous|irrelevant", '
            '"score": 0.0, "reason": "one short sentence"}]} with one entry per chunk.')
    raw = call_llm(GRADER_SYSTEM, user, json_mode=True)
    try:
        items = json.loads(raw).get("grades", [])
    except Exception:
        items = []
    by_idx = {int(g.get("index", -1)): g for g in items if isinstance(g, dict)}

    graded = []
    for i, c in enumerate(chunks):
        g = by_idx.get(i, {})
        verdict = g.get("verdict", "irrelevant")
        try:
            score = float(g.get("score", 0.0))
        except (TypeError, ValueError):
            score = 0.0
        if verdict not in {"relevant", "ambiguous", "irrelevant"}:
            verdict = "irrelevant"
        if verdict == "relevant" and score < REL_THRESHOLD:
            verdict = "ambiguous"
        if verdict == "ambiguous" and score >= RESCUE_GRADE and c["score"] >= RESCUE_SIM:
            verdict = "relevant"      # rescue: partially relevant and very similar to the question
        graded.append({**c, "verdict": verdict, "grade": score, "reason": g.get("reason", "")})
    return graded

### Retrieval status

In [12]:
def evaluate_retrieval(question: str, k: int = 5, course: str | None = None,
                       search_query: str | None = None) -> dict:
    # retrieve with the (possibly rewritten) search query, but always grade against the ORIGINAL question
    chunks = retrieve(search_query or question, k=k, course=course)
    graded = grade_chunks(question, chunks)

    relevant  = [g for g in graded if g["verdict"] == "relevant"]
    ambiguous = [g for g in graded if g["verdict"] == "ambiguous"]

    if relevant:
        status, kept = "correct", relevant
    elif ambiguous:
        status, kept = "ambiguous", []
    else:
        status, kept = "incorrect", []

    return {"question": question, "status": status,
            "kept_chunks": kept, "graded": graded,
            "confidence": max((g["grade"] for g in graded), default=0.0)}

def show(result: dict):
    print(f"Q: {result['question']}")
    print(f"Status: {result['status'].upper()}  |  kept: {len(result['kept_chunks'])}  |  best grade: {result['confidence']:.2f}")
    for g in result["graded"]:
        m = g["meta"]
        print(f"  [{g['verdict']:<10}] grade={g['grade']:.2f} sim={g['score']:.2f} | {m['course']} | {m['source']} | {m['loc']}")
        print(f"      reason: {g['reason']}")
    print()

### Test it
Expected: in-domain → `correct`, Arabic → `correct`, out-of-domain → `incorrect`.

In [13]:
tests = [
    "What is symmetric encryption?",
    "What is the internet?",
    "ما هو التشفير المتماثل؟",
    "Who won the 2022 football World Cup?",     # out of domain on purpose
]
results = []
for q in tests:
    res = evaluate_retrieval(q, k=5)
    show(res)
    results.append(res)
    time.sleep(1)

pd.DataFrame([{"question": r["question"], "status": r["status"],
               "kept": len(r["kept_chunks"]), "best_grade": round(r["confidence"], 2)} for r in results])

Q: What is symmetric encryption?
Status: CORRECT  |  kept: 3  |  best grade: 0.82
  [relevant  ] grade=0.82 sim=0.89 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 32
      reason: Explains what symmetric encryption does and how it differs from asymmetric, providing key characteristics.
  [relevant  ] grade=0.75 sim=0.89 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 21
      reason: Describes properties of symmetric encryption such as speed and common use, contributing to understanding its nature.
  [relevant  ] grade=0.78 sim=0.87 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 32
      reason: Lists reasons why symmetric encryption is used, detailing its speed and key size, which helps define it.
  [irrelevant] grade=0.09 sim=0.86 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 2
      reason: Only a table of contents with headings, no explanatory content about symmetric encryption.
  [irrelevant] grade=0.07 sim=0.85 | TM112 | TM112 Meeting12-Cryptography.pptx | sl

,question,status,kept,best_grade
0,What is symmetric encryption?,correct,3,0.82
1,What is the internet?,correct,3,0.82
2,ما هو التشفير المتماثل؟,ambiguous,0,0.50
3,Who won the 2022 football World Cup?,incorrect,0,0.00


# Part 3: Correction (rewrite the query and re-retrieve)

If the first retrieval has no `relevant` chunk:
1. The LLM **rewrites** the question into a better search query (it also translates Arabic questions to English, because the course material is in English), using the grader's notes about what was missing.
2. We retrieve again and grade again (up to `MAX_REWRITES` times).
3. Final status:
   - `verified`: at least one relevant chunk was found, so the answer will use only these.
   - `partial`: only ambiguous chunks, so the answer will be flagged as low confidence.
   - `not_found`: nothing useful, so the system will say it could not find the answer.

In [14]:
REWRITE_SYSTEM = """You rewrite user questions into better search queries for a vector search over university course material (computing, networks, cryptography, cloud computing, algorithms and data structures).
Rules:
- Output ONE standalone query in English. If the question is in Arabic, translate it faithfully.
- Keep the SAME meaning and make it short (3 to 8 words) using the core technical terms.
- Do NOT add words like "definition", "examples", "lecture" or "slide" unless the question asks for them.
- Use the evaluator notes only to choose better technical terms or synonyms, and do NOT repeat a previous query.
- Ignore evaluator notes that complain about a missing definition: keep the query about the topic itself.
Reply ONLY with JSON: {"query": "..."}"""

def rewrite_query(question: str, previous_queries: list[str], notes: str) -> str:
    user = (f"Original question: {question}\n"
            f"Previous queries (did not work): {previous_queries}\n"
            f"Evaluator notes about what was retrieved: {notes}")
    raw = call_llm(REWRITE_SYSTEM, user, json_mode=True, max_tokens=800)
    try:
        q = json.loads(raw).get("query", "").strip()
    except Exception:
        q = ""
    return q or question

In [15]:
MAX_REWRITES = 2

def _key(g):
    return (g["meta"]["source"], g["meta"]["loc"], g["text"][:80])

def corrective_retrieve(question: str, k: int = 5, course: str | None = None,
                        max_rewrites: int = MAX_REWRITES) -> dict:
    trace, tried = [], [question]
    relevant, partial = {}, {}

    def absorb(res):
        for g in res["graded"]:
            key = _key(g)
            if g["verdict"] == "relevant":
                relevant[key] = g
            elif g["verdict"] == "ambiguous" and key not in relevant:
                partial[key] = g

    res = evaluate_retrieval(question, k=k, course=course)
    trace.append({"query": question, "status": res["status"]})
    absorb(res)

    rewrites = 0
    while not relevant and rewrites < max_rewrites:
        notes = " | ".join(g["reason"] for g in res["graded"][:4] if g["reason"])
        new_q = rewrite_query(question, tried, notes)
        if new_q in tried:
            break
        rewrites += 1
        tried.append(new_q)
        res = evaluate_retrieval(question, k=k, course=course, search_query=new_q)
        trace.append({"query": new_q, "status": res["status"]})
        absorb(res)

    for key in list(partial):
        if key in relevant:
            del partial[key]

    if relevant:
        final, context = "verified", sorted(relevant.values(), key=lambda g: -g["grade"])
    elif partial:
        final, context = "partial", sorted(partial.values(), key=lambda g: -g["grade"])[:3]
    else:
        final, context = "not_found", []

    return {"question": question, "final_status": final, "context": context,
            "trace": trace, "rewrites": rewrites}

def show_corrective(r: dict):
    print(f"Q: {r['question']}")
    for i, t in enumerate(r["trace"]):
        label = "original" if i == 0 else f"rewrite {i}"
        print(f"  [{label}] {t['status']:<10} <- {t['query']}")
    print(f"  FINAL: {r['final_status'].upper()}  |  context chunks: {len(r['context'])}")
    for g in r["context"]:
        m = g["meta"]
        print(f"     - {g['verdict']:<9} {g['grade']:.2f} | {m['course']} | {m['source']} | {m['loc']}")
    print()

### Test the full correction loop
Expected: the Arabic question gets rewritten to English and becomes `verified`; the World Cup question ends as `not_found`.

In [16]:
corrective_results = []
for q in tests:
    r = corrective_retrieve(q, k=5)
    show_corrective(r)
    corrective_results.append(r)
    time.sleep(1)

pd.DataFrame([{"question": r["question"], "final": r["final_status"],
               "rewrites": r["rewrites"], "context": len(r["context"])} for r in corrective_results])

Q: What is symmetric encryption?
  [original] correct    <- What is symmetric encryption?
  FINAL: VERIFIED  |  context chunks: 3
     - relevant  0.82 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 32
     - relevant  0.78 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 32
     - relevant  0.75 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 21

Q: What is the internet?
  [original] correct    <- What is the internet?
  FINAL: VERIFIED  |  context chunks: 3
     - relevant  0.82 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 6
     - relevant  0.78 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 1
     - relevant  0.71 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 4

Q: ما هو التشفير المتماثل؟
  [original] ambiguous  <- ما هو التشفير المتماثل؟
  [rewrite 1] correct    <- symmetric encryption
  FINAL: VERIFIED  |  context chunks: 3
     - relevant  0.90 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 21
     - relevant  0.85 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 3

,question,final,rewrites,context
0,What is symmetric encryption?,verified,0,3
1,What is the internet?,verified,0,3
2,ما هو التشفير المتماثل؟,verified,1,3
3,Who won the 2022 football World Cup?,not_found,1,0


# Part 4: Answer generation (verified context only, with sources)

- `verified`: answer only from the relevant chunks, with inline citations `[1]`, `[2]`.
- `partial`: same, but the answer starts with a low-confidence warning.
- `not_found` (or the model says the context is not enough): a helpful **scope message** (what the documents cover + example questions), **no guessing**.
- The answer language follows the question (Arabic question → Arabic answer).
- Sources shown are the chunks that were actually cited: course, file, page/slide.

In [17]:
ARABIC_RE = re.compile(r"[\u0600-\u06FF]")

LOW_CONF_EN  = "⚠️ Low confidence: the documents only partly cover this question."
LOW_CONF_AR  = "⚠️ ثقة منخفضة: المستندات تغطي هذا السؤال جزئياً فقط."

ANSWER_SYSTEM = """You answer questions about university course material using ONLY the numbered context chunks provided.
Rules:
- Use only facts that appear in the context. Never add outside knowledge, even if you know the answer.
- Cite the chunks you used inline like [1] or [2][3] right after each statement.
- Answer in the same language as the question. If the question is Arabic, answer in Arabic and keep technical terms in English in parentheses where helpful.
- Be clear and concise (a short paragraph or a few bullets).
- If the context only partly answers the question, answer the part it supports and say clearly which part is not covered.
- Reply with exactly NOT_ENOUGH_INFO only when the context contains nothing relevant to the question."""

def build_context(chunks: list[dict]) -> str:
    return "\n\n".join(
        f"[{i}] ({g['meta']['course']} | {g['meta']['source']} | {g['meta']['loc']})\n{g['text']}"
        for i, g in enumerate(chunks, 1))

# ------------------------------------------------------------------
# Out-of-scope questions: no "I don't know" reply. Say what the documents cover and suggest questions they can answer.
# ------------------------------------------------------------------
_EXAMPLES_CACHE = {}

def kb_example_questions(arabic: bool, n: int = 4) -> list:
    key = (len(chunks), arabic)
    if key in _EXAMPLES_CACHE:
        return _EXAMPLES_CACHE[key]
    import random
    pool = random.Random(0).sample(chunks, min(14, len(chunks)))
    snippets = "\n".join(f"- ({c['course']}) {c['text'][:150]}".replace("\n", " ") for c in pool)
    qs = []
    try:
        raw = call_llm(
            f"You write short example questions a student could ask about the given course material. "
            f"Write {n} varied questions (different courses/topics), each under 12 words, "
            f"in {'Arabic' if arabic else 'English'}. Reply ONLY with JSON: {{\"questions\": [\"...\"]}}",
            "Material:\n" + snippets, json_mode=True, max_tokens=600)
        m = re.search(r"\{.*\}", raw or "", re.S)
        qs = [str(x).strip() for x in json.loads(m.group(0)).get("questions", []) if str(x).strip()][:n]
    except Exception:
        qs = []
    if not qs:   # fallback: use the document names as topics
        qs = sorted({Path(c["source"]).stem for c in chunks})[:n]
    _EXAMPLES_CACHE[key] = qs
    return qs

def scope_reply(question: str, is_ar: bool) -> str:
    courses = ", ".join(sorted({c["course"] for c in chunks}))
    qs = "\n".join(f"- {q}" for q in kb_example_questions(is_ar))
    if is_ar:
        return (f"📚 هذا السؤال خارج نطاق مواد الكورسات المتاحة ({courses}).\n\n"
                f"أقدر أجاوب بدقة من المستندات على أسئلة مثل:\n{qs}")
    return (f"📚 This question is outside the course material available here ({courses}).\n\n"
            f"I can answer from the documents with sources, for example:\n{qs}")

def answer_question(question: str, k: int = 5, course: str | None = None) -> dict:
    is_ar = bool(ARABIC_RE.search(question))
    not_found = lambda: scope_reply(question, is_ar)
    r = corrective_retrieve(question, k=k, course=course)

    if r["final_status"] == "not_found":
        return {"question": question, "status": "not_found", "answer": not_found(),
                "sources": [], "trace": r["trace"]}

    ctx = r["context"]
    user = f"Question: {question}\n\nContext:\n{build_context(ctx)}"
    answer = call_llm(ANSWER_SYSTEM, user, max_tokens=2000).strip().replace("【", "[").replace("】", "]")

    if "NOT_ENOUGH_INFO" in answer:
        return {"question": question, "status": "not_found", "answer": not_found(),
                "sources": [], "trace": r["trace"]}

    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer) if 1 <= int(n) <= len(ctx)})
    used = [ctx[i - 1] for i in cited] or ctx
    sources = [{"course": g["meta"]["course"], "source": g["meta"]["source"],
                "loc": g["meta"]["loc"], "relevance": round(g["grade"], 2)} for g in used]

    if r["final_status"] == "partial":
        answer = (LOW_CONF_AR if is_ar else LOW_CONF_EN) + "\n\n" + answer
    return {"question": question, "status": r["final_status"], "answer": answer,
            "sources": sources, "trace": r["trace"]}

def show_answer(res: dict):
    print(f"Q: {res['question']}")
    print(f"Status: {res['status'].upper()}  |  searches: {[t['query'] for t in res['trace']]}")
    print("\nAnswer:\n" + res["answer"])
    if res["sources"]:
        print("\nSources:")
        for s_ in res["sources"]:
            print(f"  - {s_['course']} | {s_['source']} | {s_['loc']}  (relevance {s_['relevance']})")
    print("\n" + "-" * 70 + "\n")

### Test it
Expected: grounded answers with sources for in-domain questions, an Arabic answer for the Arabic question, and the scope message for the World Cup question.

In [18]:
answer_tests = tests + ["What is Docker?"]
answers = []
for q in answer_tests:
    res = answer_question(q)
    show_answer(res)
    answers.append(res)
    time.sleep(1)

pd.DataFrame([{"question": a["question"], "status": a["status"], "sources": len(a["sources"])} for a in answers])

Q: What is symmetric encryption?
Status: VERIFIED  |  searches: ['What is symmetric encryption?']

Answer:
Symmetric encryption is a method of encrypting data in which the same secret key is used for both encryption and de‑cryption. It can handle any amount of data (even when the final size is unknown, such as a live telephone call) and is able to run at very high speeds on modest hardware, often using small keys that are quick to generate and use[1†L1-L4][2†L1-L3].

Sources:
  - TM112 | TM112 Meeting12-Cryptography.pptx | slide 32  (relevance 0.82)
  - TM112 | TM112 Meeting12-Cryptography.pptx | slide 21  (relevance 0.78)
  - TM112 | TM112 Meeting12-Cryptography.pptx | slide 32  (relevance 0.75)

----------------------------------------------------------------------

Q: What is the internet?
Status: VERIFIED  |  searches: ['What is the internet?']

Answer:
The internet is a “network of networks” – a massive collection of many different local, campus, corporate and other networks that 

,question,status,sources
0,What is symmetric encryption?,verified,3
1,What is the internet?,verified,2
2,ما هو التشفير المتماثل؟,verified,3
3,Who won the 2022 football World Cup?,not_found,0
4,What is Docker?,verified,2


## Dataset setup (for the `analyze_data` tool)
Tables such as `per_class_test.csv` (per-class results of the YOLO models) are **not** turned into text chunks. They stay as tables, and the agent analyzes them with pandas.
Put the file in `/content/datasets/` (Files panel, drag and drop) or run the cell below to upload it.

In [19]:
from pathlib import Path
import shutil

DATASET_DIR = Path("/content/datasets"); DATASET_DIR.mkdir(exist_ok=True)
TARGET = DATASET_DIR / "per_class_test.csv"

if not TARGET.exists() and Path("/content/per_class_test.csv").exists():
    shutil.copy("/content/per_class_test.csv", TARGET)

if not TARGET.exists():
    print("per_class_test.csv not found. Upload it now.")
    try:
        from google.colab import files
        for fname in files.upload():
            shutil.move(fname, DATASET_DIR / fname)
    except Exception as e:
        print("Upload skipped:", e, "-> copy per_class_test.csv into /content/datasets/ with the Files panel.")

print("CSV files:", [f"{f.name} ({f.stat().st_size/1e6:.2f} MB)" for f in DATASET_DIR.glob("*.csv")])
print("per_class_test ready ✅" if TARGET.exists() else "⚠️ per_class_test.csv is still missing: the data questions will fail.")


CSV files: ['per_class_test.csv (0.00 MB)']
per_class_test ready ✅


# Part 5: Tools
In Parts 1-4 the pipeline was **fixed**: retrieve → grade → rewrite → answer. In an Agentic RAG system the **agent decides** what to do. For that it needs tools.

| Tool | What it does | Built on |
|---|---|---|
| `search_kb` | Self-correcting search of the documents | `corrective_retrieve` (Parts 2-3) |
| `summarize` | Summary of a file, a whole course or a topic | ChromaDB chunks + LLM |
| `compare` | Retrieves two concepts separately, then compares them | `corrective_retrieve` x2 + LLM |
| `analyze_data` | Count / filter / aggregate tabular data (CSV files + stats about the knowledge base) | LLM writes pandas, runs in a restricted sandbox |
| `calculator` | Exact arithmetic | safe AST evaluator (no `eval`) |

Every fact a tool returns gets an **evidence id** `[n]`, so the final answer can cite it and the UI can show the real sources.

In [20]:
import ast, math, operator, builtins
import numpy as np

# ------------------------------------------------------------------
# Robust LLM wrapper
#  1) Reasoning models (gpt-oss) sometimes spend all their tokens thinking and return an empty JSON
#     -> Groq error "json_validate_failed": give JSON calls more room and fall back to plain text.
#  2) On a rate limit (429) switch to the next available model instead of failing / waiting.
# ------------------------------------------------------------------
if "_base_call_llm" not in globals():
    _base_call_llm = call_llm

def _rotate_model() -> bool:
    global MODEL
    pool = [m for m in PREFERRED if m in available and m != MODEL]
    if not pool:
        return False
    MODEL = pool[0]
    print(f"Rate limit reached: switched to {MODEL}")
    return True

def _call_once(system, user, json_mode, temperature, max_tokens, retries):
    if not json_mode:
        return _base_call_llm(system, user, False, temperature, max_tokens, retries)
    big = max(max_tokens, 2500)
    try:
        return _base_call_llm(system, user, True, temperature, big, 1)
    except Exception as e:
        if "json_validate_failed" not in str(e) and "400" not in str(e):      # e.g. rate limit -> normal retries
            return _base_call_llm(system, user, True, temperature, big, retries)
    return _base_call_llm(system + "\nReply ONLY with one valid JSON object and nothing else.", user,
                          False, temperature, max(max_tokens, 3500), retries)

def call_llm(system: str, user: str, json_mode: bool = False,
             temperature: float = 0.0, max_tokens: int = 2000, retries: int = 3) -> str:
    for attempt in range(2):                       # the 2nd pass runs after a model switch
        try:
            return _call_once(system, user, json_mode, temperature, max_tokens, retries)
        except Exception as e:
            is_rate = "429" in str(e) or "rate_limit" in str(e).lower()
            if is_rate and attempt == 0 and _rotate_model():
                continue
            raise

# Arabic question -> Arabic "not found" message (it was English before)
NOT_FOUND_AR = "عذرًا، ليس لدي أي معلومات عن هذا السؤال في المستندات."

COURSES = sorted({c["course"] for c in chunks})

# ------------------------------------------------------------------
# Evidence registry: every fact the agent collects gets an id [n],
# so the final answer can cite it and we can show the real sources.
# ------------------------------------------------------------------
def new_run():
    return {"evidence": [], "steps": [], "plan": None}

def add_evidence(run, kind, label, text, **meta):
    key = (kind, label, text[:80])
    for e in run["evidence"]:
        if e["key"] == key:
            return e["id"]
    eid = len(run["evidence"]) + 1
    run["evidence"].append({"id": eid, "key": key, "kind": kind,
                            "label": label, "text": text, "meta": meta})
    return eid

def _json(raw, default):
    """Parse JSON from an LLM reply, tolerating extra text around it."""
    try:
        return json.loads(raw)
    except Exception:
        m = re.search(r"\{.*\}", raw or "", re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return default

# ------------------------------------------------------------------
# Tool 1: search_kb  (= the Corrective RAG from Parts 2-3)
# ------------------------------------------------------------------
def tool_search_kb(args, run):
    query = str(args.get("query", "")).strip()
    if not query:
        return "ERROR: 'query' is required."
    course = args.get("course")
    if course not in COURSES:
        course = None
    r = corrective_retrieve(query, k=5, course=course, max_rewrites=1)   # the agent loop is the outer correction
    tried = [t["query"] for t in r["trace"]]
    if r["final_status"] == "not_found":
        return f"NO RELEVANT RESULTS for '{query}' (queries tried: {tried}). Try other terms or another course."
    conf = "verified" if r["final_status"] == "verified" else "partial"
    lines = []
    for g in r["context"][:4]:
        m = g["meta"]
        label = f"{m['course']} | {m['source']} | {m['loc']}"
        eid = add_evidence(run, "doc", label, g["text"], conf=conf, grade=round(g["grade"], 2))
        lines.append(f"[{eid}] ({label}) {g['text'][:500]}")
    note = " NOTE: 'partial' means related but not a perfect match; it is usually enough, do not repeat this search." if conf == "partial" else ""
    return f"Found {len(lines)} chunk(s), confidence={conf}, queries tried: {tried}.{note}\n" + "\n".join(lines)

# ------------------------------------------------------------------
# Tool 2: summarize  (a file, a whole course, or a topic)
# ------------------------------------------------------------------
def tool_summarize(args, run):
    target = str(args.get("target", "")).strip()
    focus = str(args.get("focus", "")).strip()
    if not target:
        return "ERROR: 'target' is required (file name, course code or topic)."
    tl = target.lower()
    sources = sorted({c["source"] for c in chunks})
    src = next((s for s in sources if tl == s.lower()), None) or next((s for s in sources if tl in s.lower()), None)

    if target.upper() in COURSES:
        items = [c for c in chunks if c["course"] == target.upper()]
        label = target.upper()
    elif src:
        items = [c for c in chunks if c["source"] == src]
        label = src
    else:
        items = [{"text": r["text"], **r["meta"]} for r in retrieve(target, k=8)]
        label = f"topic: {target}"
    if not items:
        return f"No documents found for '{target}'."

    MAX_PIECES = 10                      # sample evenly so long files still fit in one prompt
    if len(items) > MAX_PIECES:
        idx = sorted(set(np.linspace(0, len(items) - 1, MAX_PIECES).astype(int).tolist()))
        items = [items[i] for i in idx]
    text = "\n\n".join(f"({c['source']} | {c['loc']}) {c['text'][:700]}" for c in items)
    summary = call_llm(
        "You summarize university course material. Use ONLY the provided text. "
        "Write 5 to 8 concise bullet points covering the main ideas. Write in English.",
        f"Target: {label}\nFocus: {focus or 'general overview'}\n\nText:\n{text}",
        max_tokens=1200).strip()
    srcs = sorted({f"{c['course']} | {c['source']}" for c in items})
    eid = add_evidence(run, "summary", f"Summary of {label}", summary, conf="verified", sources=srcs)
    return f"[{eid}] Summary of {label}:\n{summary}"

# ------------------------------------------------------------------
# Tool 3: compare  (retrieves each side separately, then compares)
# ------------------------------------------------------------------
COMPARE_SYSTEM = """You compare two items using ONLY the numbered evidence provided.
Write short bullets: key similarities, key differences, and (if asked) which fits a given aspect better.
Cite the evidence ids like [3] after each statement. If one side is not covered for a point, say so. Do not use outside knowledge."""

def tool_compare(args, run):
    a, b = str(args.get("item_a", "")).strip(), str(args.get("item_b", "")).strip()
    aspect = str(args.get("aspect", "")).strip()
    if not a or not b:
        return "ERROR: 'item_a' and 'item_b' are required."
    course = args.get("course") if args.get("course") in COURSES else None
    sides = {}
    for name, item in (("A", a), ("B", b)):
        r = corrective_retrieve(item, k=4, course=course, max_rewrites=0)
        conf = "verified" if r["final_status"] == "verified" else "partial"
        sides[name] = []
        for g in r["context"][:3]:
            m = g["meta"]
            eid = add_evidence(run, "doc", f"{m['course']} | {m['source']} | {m['loc']}", g["text"],
                               conf=conf, grade=round(g["grade"], 2))
            sides[name].append((eid, g["text"]))
    ctx = "\n\n".join(
        f"SIDE {n} ({a if n == 'A' else b}):\n" + "\n".join(f"[{i}] {t[:600]}" for i, t in sides[n])
        for n in "AB" if sides[n])
    missing = [n for n in "AB" if not sides[n]]
    if missing:
        return f"INCOMPLETE: no relevant material found for side(s) {missing}.\n{ctx}"
    cmp_txt = call_llm(COMPARE_SYSTEM,
                       f"A = {a}\nB = {b}\nAspect: {aspect or 'general comparison'}\n\n{ctx}",
                       max_tokens=1200).strip()
    refs = [i for n in sides for i, _ in sides[n]]
    eid = add_evidence(run, "comparison", f"Comparison: {a} vs {b}", cmp_txt, conf="verified", refs=refs)
    return f"[{eid}] Comparison of '{a}' vs '{b}':\n{cmp_txt}"

# ------------------------------------------------------------------
# Tool 4: analyze_data  (LLM writes a small pandas snippet, we run it in a sandbox)
# Datasets: every non-derived CSV in the data folder + 'kb_chunks' (stats about the knowledge base itself)
# ------------------------------------------------------------------
import warnings, difflib
EXTRA_DATA_DIRS = [Path("/content/datasets")]     # put big CSV files here (not ingested into the knowledge base)

DATASETS = {"kb_chunks": pd.DataFrame([{"course": c["course"], "source": c["source"], "loc": c["loc"],
                                        "n_chars": len(c["text"])} for c in chunks])}

def _load_table(f):
    df = pd.read_csv(f, on_bad_lines="skip", low_memory=False)
    first = df.columns[0]                                  # name the unlabeled first column (e.g. "Unnamed: 0") so the LLM can use it
    if str(first).startswith("Unnamed") and not pd.api.types.is_numeric_dtype(df[first]):
        df = df.rename(columns={first: "class"})
    for c in df.columns:                                   # parse date columns so the agent can group by month/year
        if "date" in str(c).lower() and not pd.api.types.is_numeric_dtype(df[c]):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                parsed = pd.to_datetime(df[c], errors="coerce")
            if parsed.notna().mean() > 0.8:
                df[c] = parsed
    return df

_seen_files = set()
_candidates = list(DATA_DIR.rglob("*.csv"))
for _d in EXTRA_DATA_DIRS:
    if _d.exists():
        _candidates += list(_d.glob("*.csv"))                # non-recursive: avoids /content/sample_data
for _f in sorted(_candidates):
    if _f.stem.endswith(("_slides", "_notes")) or _f.resolve() in _seen_files:
        continue                                             # derived from a .pptx, or duplicate
    _seen_files.add(_f.resolve())
    name = re.sub(r"_?cleaned(_\d+)?$", "", _f.stem.lower()) or _f.stem.lower()
    try:
        DATASETS[name] = _load_table(_f)
    except Exception as _e:
        print("Could not load", _f.name, "->", _e)
print("Datasets available to analyze_data:")
for _k, _v in DATASETS.items():
    print(f"  {_k}: {_v.shape[0]:,} rows x {_v.shape[1]} columns")

def resolve_dataset(name):
    """Tolerate small naming mistakes by the LLM (case, underscores, partial names)."""
    if name in DATASETS:
        return name
    norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
    n = norm(name)
    for k in DATASETS:
        if norm(k) == n:
            return k
    if n:
        for k in DATASETS:
            if n in norm(k) or norm(k) in n:
                return k
        m = difflib.get_close_matches(n, [norm(k) for k in DATASETS], n=1, cutoff=0.5)
        if m:
            return next(k for k in DATASETS if norm(k) == m[0])
    return None

_SCHEMA_CACHE = {}
def dataset_schema(name):
    """Columns, dtypes, sample rows and the values of low-cardinality text columns (so the LLM filters correctly)."""
    if name in _SCHEMA_CACHE:
        return _SCHEMA_CACHE[name]
    df = DATASETS[name]
    cats = []
    for c in df.columns:
        if not (pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_datetime64_any_dtype(df[c])):
            u = df[c].dropna().unique()
            if len(u) <= 12:
                cats.append(f"  {c}: {[str(x) for x in u]}")
    schema = (f"rows: {len(df):,}\ncolumns: {list(df.columns)}\ndtypes: {dict(df.dtypes.astype(str))}\n"
              f"sample rows:\n{df.head(2).T.to_string()[:1500]}\n"
              + ("categorical columns and their values:\n" + "\n".join(cats) if cats else ""))
    _SCHEMA_CACHE[name] = schema
    return schema

def dataset_card(name, maxc=1200):
    df = DATASETS[name]
    return f"   - {name} ({len(df):,} rows x {df.shape[1]} cols): " + ", ".join(map(str, df.columns))[:maxc]

_BAD_NAMES = {"exec", "eval", "open", "compile", "__import__", "input", "globals", "locals", "getattr",
              "setattr", "delattr", "vars", "breakpoint", "os", "sys", "subprocess", "exit", "quit"}
_BAD_ATTRS = {"to_csv", "to_pickle", "to_excel", "to_json", "to_sql", "to_parquet", "to_feather",
              "to_hdf", "to_html", "to_latex", "to_clipboard", "eval", "system"}
_SAFE_BUILTINS = {n: getattr(builtins, n) for n in
                  ["len", "sum", "min", "max", "round", "sorted", "abs", "range", "list", "dict", "set",
                   "str", "int", "float", "bool", "enumerate", "zip", "tuple", "any", "all", "reversed",
                   "isinstance", "map", "filter"]}

def _check_code(code: str):
    tree = ast.parse(code)
    for n in ast.walk(tree):
        if isinstance(n, (ast.Import, ast.ImportFrom)):
            raise ValueError("imports are not allowed")
        if isinstance(n, ast.Attribute) and (n.attr.startswith("_") or n.attr.startswith("read_") or n.attr in _BAD_ATTRS):
            raise ValueError(f"attribute '{n.attr}' is not allowed")
        if isinstance(n, ast.Name) and (n.id in _BAD_NAMES or n.id.startswith("__")):
            raise ValueError(f"name '{n.id}' is not allowed")
    return tree

def _run_pandas(code: str, df):
    tree = _check_code(code)
    env = {"__builtins__": _SAFE_BUILTINS, "pd": pd, "np": np, "df": df.copy()}
    exec(compile(tree, "<agent>", "exec"), env)
    if "result" not in env:
        raise ValueError("the code did not assign a variable named `result`")
    res = env["result"]
    if isinstance(res, (pd.DataFrame, pd.Series)):
        return res.head(25).to_string()[:1500]
    if isinstance(res, np.generic):
        res = res.item()
    if isinstance(res, (dict, list, tuple)):                 # clean output: no np.float64(...) wrappers
        try:
            return json.dumps(res, default=lambda o: o.item() if hasattr(o, "item") else str(o),
                              ensure_ascii=False)[:1500]
        except Exception:
            pass
    return str(res)[:1500]

DATA_SYSTEM = """You write a short pandas snippet that answers a question about a DataFrame named `df`.
Rules: no imports (pd and np are already available), no file or network access, no printing.
Column names may contain spaces and parentheses, so always write df["Column Name"] with the exact name.
Use the categorical values listed in the schema exactly as written. For a rate or percentage use the mean of a 0/1 column or value_counts(normalize=True).
Round numbers to 2 decimals and keep tables small (at most 10 rows).
Assign the final answer to a variable named `result` (a number, string, list, Series or DataFrame).
Reply ONLY with JSON: {"code": "..."}"""

def tool_analyze_data(args, run):
    question = str(args.get("question", "")).strip()
    name = resolve_dataset(str(args.get("dataset", "")).strip())
    if name is None:
        return f"ERROR: unknown dataset '{args.get('dataset')}'. Available datasets: {list(DATASETS)}"
    if not question:
        return "ERROR: 'question' is required."
    df = DATASETS[name]
    schema = dataset_schema(name)
    err, code, result = "", "", None
    for attempt in range(2):
        user = f"Question: {question}\n\nDataFrame schema:\n{schema}"
        if err:
            user += f"\n\nYour previous code failed with: {err}\nPrevious code:\n{code}\nFix it."
        code = str(_json(call_llm(DATA_SYSTEM, user, json_mode=True, max_tokens=800), {}).get("code", "")).strip()
        try:
            result = _run_pandas(code, df)
            break
        except Exception as e:
            err = f"{e.__class__.__name__}: {e}"
    if result is None:
        return f"TOOL ERROR: could not analyze '{name}' ({err})."
    eid = add_evidence(run, "data", f"Data analysis on '{name}'",
                       f"Question: {question}\nCode: {code}\nResult: {result}", conf="verified")
    return f"[{eid}] Result from dataset '{name}': {result}"

# ------------------------------------------------------------------
# Tool 5: calculator  (safe arithmetic, no eval)
# ------------------------------------------------------------------
_BINOPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
           ast.Pow: operator.pow, ast.Mod: operator.mod, ast.FloorDiv: operator.floordiv}
_UNOPS = {ast.USub: operator.neg, ast.UAdd: operator.pos}
_FUNCS = {"sqrt": math.sqrt, "log": math.log, "log2": math.log2, "log10": math.log10, "sin": math.sin,
          "cos": math.cos, "tan": math.tan, "abs": abs, "round": round, "min": min, "max": max,
          "ceil": math.ceil, "floor": math.floor, "factorial": math.factorial}
_CONSTS = {"pi": math.pi, "e": math.e}

def safe_eval(expr: str):
    expr = re.sub(r"(?<=\d),(?=\d{3}\b)", "", expr)                  # 1,000 -> 1000
    expr = expr.replace("^", "**").replace("×", "*").replace("÷", "/")
    def ev(n):
        if isinstance(n, ast.Expression):
            return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
            return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _BINOPS:
            l, r = ev(n.left), ev(n.right)
            if isinstance(n.op, ast.Pow) and abs(r) > 1000:
                raise ValueError("exponent too large")
            return _BINOPS[type(n.op)](l, r)
        if isinstance(n, ast.UnaryOp) and type(n.op) in _UNOPS:
            return _UNOPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and isinstance(n.func, ast.Name) and n.func.id in _FUNCS and not n.keywords:
            return _FUNCS[n.func.id](*[ev(a) for a in n.args])
        if isinstance(n, ast.Name) and n.id in _CONSTS:
            return _CONSTS[n.id]
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr.strip(), mode="eval"))

def tool_calculator(args, run):
    expr = str(args.get("expression", "")).strip()
    if not expr:
        return "ERROR: 'expression' is required."
    try:
        val = safe_eval(expr)
    except Exception as e:
        return f"CALC ERROR: {e}. Use only numbers, + - * / ** %, parentheses and functions like sqrt, log2, round."
    if isinstance(val, float):
        val = float(f"{val:.10g}")
    eid = add_evidence(run, "calc", f"Calculator: {expr}", f"{expr} = {val}", conf="verified")
    return f"[{eid}] {expr} = {val}"

# ------------------------------------------------------------------
TOOLS = {"search_kb": tool_search_kb, "summarize": tool_summarize, "compare": tool_compare,
         "analyze_data": tool_analyze_data, "calculator": tool_calculator}

def tools_description() -> str:
    return f"""Available tools (action_input must be a JSON object):
1. search_kb      {{"query": str, "course": optional, one of {COURSES}}}
   Search the course documents (self-correcting retrieval). Use this for definitions and factual questions. One focused sub-question per call. Write queries in English.
   Set "course" ONLY if the user restricted the course or named one. Otherwise leave it out so all courses are searched.
2. summarize      {{"target": str, "focus": optional str}}
   Use ONLY when the user asks for a summary / overview / main topics of a file, a course or a topic. NEVER for a definition or a factual question (use search_kb).
3. compare        {{"item_a": str, "item_b": str, "aspect": optional str, "course": optional}}
   Retrieve material for two concepts separately and compare them. Leave "course" out unless the user named one.
4. analyze_data   {{"dataset": one of {list(DATASETS)}, "question": str}}
   Count / filter / aggregate / rank tabular data (per-class model results, scores, n_instances, counts, averages, percentages, rankings...). Write the question in plain English.
   Datasets:
{chr(10).join(dataset_card(k) for k in DATASETS)}
   ('kb_chunks' = statistics about the knowledge base itself: chunks per course/file, text length.)
5. calculator     {{"expression": str}}
   Exact arithmetic. ALWAYS use it for any calculation instead of computing in your head."""

# quick self-test of the tools that do not need the LLM
assert safe_eval("2**10 + 3*4") == 1036
assert safe_eval("0.12 * 24 * 30 * 3") - 259.2 < 1e-9
assert safe_eval("sqrt(144) + log2(8)") == 15
for bad in ["__import__('os').system('ls')", "open('x')", "(1).__class__"]:
    try:
        safe_eval(bad); raise SystemExit("calculator is unsafe!")
    except SystemExit:
        raise
    except Exception:
        pass
print("Tools ready:", list(TOOLS))

Datasets available to analyze_data:
  kb_chunks: 296 rows x 4 columns
  per_class_test: 6 rows x 5 columns
Tools ready: ['search_kb', 'summarize', 'compare', 'analyze_data', 'calculator']


In [21]:
# Guard + preview: the analyze_data demo questions use per_class_test.csv
if "per_class_test" not in DATASETS:
    raise RuntimeError("per_class_test is not loaded. Run the 'Dataset setup' cell, then re-run Part 5.")
_df = DATASETS["per_class_test"]
print("per_class_test loaded ✅", _df.shape, "| columns:", list(_df.columns))
display(_df.head(10))
print("If a column name differs from the questions in the test cells, edit the questions to match it.")

per_class_test loaded ✅ (6, 5) | columns: ['class', 'YOLOv8s', 'YOLO11s', 'YOLOv8n', 'n_instances']


,class,YOLOv8s,YOLO11s,YOLOv8n,n_instances
0,alligator,0.142,0.150,0.150,577
1,block,0.593,0.642,0.598,47
2,crack,0.599,0.626,0.584,93
3,longitudinal,0.090,0.095,0.078,311
4,pothole,0.382,0.391,0.375,909
5,transverse,0.362,0.350,0.358,67


If a column name differs from the questions in the test cells, edit the questions to match it.


# Part 6: Planner + Agent loop
```
question
   │
   ▼
PLANNER ── needs tools? how complex? which steps?        (greetings are answered directly)
   │
   ▼
AGENT LOOP (max N steps):  decide next action → run tool → observe result → decide again
   │                        (retries with new wording when a search fails, skips duplicates, stops when enough evidence)
   ▼
FINAL ANSWER ── only from collected evidence, cites [n], says which part is missing instead of guessing
```

In [22]:
from collections import Counter

# ------------------------------------------------------------------
# 1) PLANNER: understand the question and decide the strategy
# ------------------------------------------------------------------
PLANNER_SYSTEM = """You are the planning module of a knowledge assistant over university course documents.
Analyse the user's question and decide the strategy BEFORE any retrieval.

{TOOLS}

Rules:
- needs_tools = false ONLY for greetings, thanks, or questions about the assistant itself. Everything else needs tools.
- Split multi-part questions into separate steps (one search per sub-question). Never plan more than 5 steps.
- If the question asks to compare two things, use the compare tool (or one search per side).
- Any arithmetic (costs, totals, percentages, conversions) must use the calculator, in its own step AFTER the facts it needs are found.
- Questions about numbers in a dataset (per-class results of the YOLO models, scores, n_instances, counts, averages, percentages, rankings, "how many files/chunks/rows") use analyze_data with the matching dataset.
- Definitions and factual questions ("what is X", "how does X work") ALWAYS use search_kb, never summarize.
- Only requests for an overview / summary / main topics of a file, course or topic use summarize.
- Do not set a "course" in tool inputs unless the user named or restricted a course.
- If the question is Arabic, still write tool queries in English (the documents are English).
- If the question adds a context or condition (e.g. "for cloud computing", "in networks"), add a separate search_kb step for that context.
- Keep it minimal: a simple factual question is ONE search_kb step.

Reply ONLY with JSON:
{"complexity": "simple|multi_step", "needs_tools": true,
 "plan": [{"step": 1, "goal": "short goal", "tool": "tool name", "input": {...}}]}"""

def make_plan(question: str, course: str | None = None) -> dict:
    system = PLANNER_SYSTEM.replace("{TOOLS}", tools_description())
    user = f"Question: {question}" + (f"\nThe user restricted the search to course: {course}" if course else "")
    d = _json(call_llm(system, user, json_mode=True, max_tokens=1500), {})
    steps = []
    for s in d.get("plan", []) if isinstance(d.get("plan"), list) else []:
        if isinstance(s, dict) and s.get("tool") in TOOLS:
            steps.append({"step": len(steps) + 1, "goal": str(s.get("goal", "")),
                          "tool": s["tool"], "input": s.get("input") if isinstance(s.get("input"), dict) else {}})
    needs = d.get("needs_tools", True) is not False
    if needs and not steps:                               # planner failed -> safe default
        steps = [{"step": 1, "goal": "search the knowledge base", "tool": "search_kb", "input": {"query": question}}]
    return {"complexity": d.get("complexity", "simple"), "needs_tools": needs, "plan": steps[:5]}

# ------------------------------------------------------------------
# 2) AGENT LOOP: decide -> act (tool) -> observe -> decide ...
# ------------------------------------------------------------------
AGENT_SYSTEM = """You are the executor of a knowledge assistant. You get a question, a plan and the progress so far.
At each turn choose exactly ONE next action.

{TOOLS}
6. final_answer  {}   Choose this when the evidence collected covers every part of the question, or when more searching will not help.

Rules:
- Follow the plan, but adapt: if a search found nothing relevant, try ONE differently-worded query or another course; then move on.
- Use numbers found in the evidence when calling the calculator.
- Never repeat an identical action. Do not search for things you already have.
- Do not add a "course" to tool inputs unless the user named or restricted one.
- A search with confidence=partial is acceptable (the chunks are related). Do NOT repeat a search for the same sub-question: move on or choose final_answer.
- If two searches in a row found nothing relevant, choose final_answer: the knowledge base does not cover it. Never answer from general knowledge.
- Stop as soon as everything needed is collected.

Reply ONLY with JSON: {"thought": "one short sentence", "action": "<tool name or final_answer>", "action_input": {...}}"""

def decide_next(question, plan, scratch, step_no, max_steps):
    plan_txt = "\n".join(f"{s['step']}. [{s['tool']}] {s['goal']} {json.dumps(s['input'], ensure_ascii=False)}"
                         for s in plan["plan"]) or "(no plan)"
    done = "\n".join(
        f"Step {i + 1}: action={s['action']} input={json.dumps(s['input'], ensure_ascii=False)}\n"
        f"Observation: {s['observation'][:1200]}" for i, s in enumerate(scratch)) or "(nothing yet)"
    user = (f"Question: {question}\n\nPlan:\n{plan_txt}\n\nProgress so far:\n{done}\n\n"
            f"This is step {step_no} of at most {max_steps}. Choose the next action.")
    d = _json(call_llm(AGENT_SYSTEM.replace("{TOOLS}", tools_description()), user, json_mode=True, max_tokens=1000), {})
    inp = d.get("action_input")
    return str(d.get("thought", "")), str(d.get("action", "final_answer")), inp if isinstance(inp, dict) else {}

# ------------------------------------------------------------------
# 3) FINAL ANSWER: only from collected evidence, with citations
# ------------------------------------------------------------------
FINAL_SYSTEM = """You write the final answer for a university course assistant using ONLY the numbered evidence provided.
Rules:
- Use only facts that appear in the evidence. Never add outside knowledge, even if you know the answer.
- Cite evidence ids inline using ONLY plain square brackets with the number, like [1] or [2][3], right after each statement. Never use any other citation format. Calculator and data results can be cited too.
- Answer EVERY part of the question. If a part is not covered by the evidence, say clearly which part is missing instead of guessing.
- For comparisons, use a short structure (bullets or a small table). Be concise.
- Answer in the same language as the question. If it is Arabic, answer in Arabic and keep technical terms in English in parentheses where helpful.
- If the evidence has nothing relevant to the question, reply with exactly: NOT_ENOUGH_INFO"""

def write_final(question: str, run: dict) -> dict:
    is_ar = bool(ARABIC_RE.search(question))
    not_found = lambda: scope_reply(question, is_ar)
    ev = run["evidence"]
    if not ev:
        return {"status": "not_found", "answer": not_found(), "sources": []}

    block = "\n\n".join(f"[{e['id']}] ({e['kind']}: {e['label']})\n{e['text'][:900]}" for e in ev)
    answer = call_llm(FINAL_SYSTEM, f"Question: {question}\nAnswer language: {'Arabic' if is_ar else 'English'}\n\nEvidence:\n{block}", max_tokens=2500).strip()
    if "NOT_ENOUGH_INFO" in answer:
        return {"status": "not_found", "answer": not_found(), "sources": []}

    answer = re.sub(r"【(\d+)[^】]*】", r"[\1]", answer)      # normalize odd citation formats like 【1†L2-L4】
    by_id = {e["id"]: e for e in ev}
    cited = {int(n) for n in re.findall(r"\[(\d+)\]", answer) if int(n) in by_id} or set(by_id)
    for i in list(cited):                                  # a comparison is backed by the chunks it used
        cited |= set(by_id[i]["meta"].get("refs", []))
    sources = []
    for i in sorted(cited):
        e = by_id[i]
        if e["kind"] == "comparison":
            continue
        if e["kind"] == "summary":
            for s in e["meta"].get("sources", []):
                sources.append({"id": i, "type": "summary", "label": f"{s} (summary)"})
        else:
            sources.append({"id": i, "type": e["kind"], "label": e["label"]})

    strong = any(e["kind"] != "doc" or e["meta"].get("conf") == "verified" for e in ev)
    status = "answered" if strong else "low_confidence"
    if not strong:
        answer = (LOW_CONF_AR if is_ar else LOW_CONF_EN) + "\n\n" + answer
    return {"status": status, "answer": answer, "sources": sources}

# ------------------------------------------------------------------
# 4) Put it together
# ------------------------------------------------------------------
CHAT_SYSTEM = ("You are a friendly assistant that answers questions about university course documents "
               "(TM112, TM275, M269). Reply briefly in the user's language and invite them to ask a course question.")

def run_agent(question: str, course: str | None = None, max_steps: int = 6) -> dict:
    t0 = time.time()
    run = new_run()
    plan = make_plan(question, course)
    run["plan"] = plan

    if not plan["needs_tools"]:
        return {"question": question, "status": "chat", "answer": call_llm(CHAT_SYSTEM, question, max_tokens=400).strip(),
                "sources": [], "plan": plan, "steps": [], "seconds": round(time.time() - t0, 1)}

    scratch, seen, misses = [], set(), 0
    for step_no in range(1, max_steps + 1):
        thought, action, inp = decide_next(question, plan, scratch, step_no, max_steps)
        if action == "final_answer":
            run["steps"].append({"thought": thought, "action": "final_answer", "input": {}, "observation": ""})
            break
        if course and action in ("search_kb", "compare") and not inp.get("course"):
            inp["course"] = course                      # respect the UI course filter
        sig = (action, json.dumps(inp, sort_keys=True, ensure_ascii=False))
        if action not in TOOLS:
            obs = f"ERROR: unknown tool '{action}'. Use one of {list(TOOLS)} or final_answer."
        elif sig in seen:
            obs = "Duplicate action skipped. Choose a different action or final_answer."
        else:
            seen.add(sig)
            try:
                obs = TOOLS[action](inp, run)
            except Exception as e:
                obs = f"TOOL ERROR ({e.__class__.__name__}): {str(e)[:200]}"
        rec = {"thought": thought, "action": action, "input": inp, "observation": obs}
        scratch.append(rec)
        run["steps"].append(rec)
        misses = misses + 1 if obs.startswith(("NO RELEVANT RESULTS", "INCOMPLETE")) else 0
        if misses >= 2:                                   # the knowledge base does not cover it: stop early
            run["steps"].append({"thought": "Two searches in a row found nothing relevant, stopping.",
                                 "action": "final_answer", "input": {}, "observation": ""})
            break

    fin = write_final(question, run)
    return {"question": question, **fin, "plan": plan, "steps": run["steps"],
            "seconds": round(time.time() - t0, 1)}

def show_agent(res: dict):
    print(f"Q: {res['question']}")
    print(f"Status: {res['status'].upper()}  |  complexity: {res['plan']['complexity']}  |  {res['seconds']}s")
    print("\nPLAN:")
    for s in res["plan"]["plan"]:
        print(f"  {s['step']}. [{s['tool']}] {s['goal']}")
    print("\nEXECUTION:")
    for i, s in enumerate(res["steps"], 1):
        print(f"  {i}. {s['action']} {json.dumps(s['input'], ensure_ascii=False)}")
        if s["thought"]: print(f"       thought: {s['thought']}")
        if s["observation"]: print(f"       result : {s['observation'][:160].replace(chr(10), ' ')}...")
    print("\nANSWER:\n" + res["answer"])
    if res["sources"]:
        print("\nSOURCES:")
        for s in res["sources"]:
            print(f"  [{s['id']}] {s['label']}")
    print("\n" + "-" * 70 + "\n")

### Test the agent
Expected: simple question → 1 search; comparison → `compare`; cost question → `search_kb` + `calculator`; chunk-count question → `analyze_data`; Arabic question answered in Arabic; World Cup question → scope message.

In [23]:
agent_tests = [
    "What is symmetric encryption?",                                    # simple -> 1 search
    "Compare symmetric and asymmetric encryption and tell me which is better for cloud computing.",   # compare / multi-step
    "If a cloud server costs 0.12 dollars per hour, how much do 3 servers cost for 30 days running 24/7, and what is cloud computing?",  # search + calculator
    "How many chunks does each course have in the knowledge base?",     # analyze_data
    "Which class has the highest number of instances?",   # expected: pothole (909 instances)
    "Which YOLO model has the highest average score across all classes, and how much is 10% of that average?",                     # expected: YOLO11s, mean 0.376, 10% = 0.0376
    "ما الفرق بين الشبكات السلكية واللاسلكية؟",                          # Arabic multi-part
    "Who won the 2022 football World Cup?",                             # out of domain -> not found
]
agent_results = []
for q in agent_tests:
    try:
        res = run_agent(q)
        show_agent(res)
        agent_results.append(res)
        time.sleep(1)
    except Exception as e:
        if "rate_limit" in str(e).lower() or "429" in str(e):
            print(f"Rate limit reached for {MODEL}. Attempting to switch model...")
            # Dynamically switch model if we have a backup
            backups = [m for m in PREFERRED if m != MODEL]
            if backups:
                MODEL = backups[0]
                print(f"Switched to backup model: {MODEL}. Retrying query...")
                try:
                    res = run_agent(q)
                    show_agent(res)
                    agent_results.append(res)
                    time.sleep(1)
                except Exception as retry_err:
                    print(f"Could not complete even with backup model: {retry_err}")
                    break
            else:
                print("No backup models available. Please wait for the daily rate limit to reset.")
                break
        else:
            print(f"Error processing '{q}': {e}")
            break

Q: What is symmetric encryption?
Status: NOT_FOUND  |  complexity: simple  |  20.6s

PLAN:
  1. [search_kb] Find definition of symmetric encryption

EXECUTION:
  1. final_answer {}

ANSWER:
📚 This question is outside the course material available here (M269, TM112, TM275).

I can answer from the documents with sources, for example:
- Why use Docker volumes for source code?
- What’s the difference between IaaS, PaaS, and SaaS?
- How does merge sort achieve O(n log n) average case?
- What is the key length of SHA-1?

----------------------------------------------------------------------

Q: Compare symmetric and asymmetric encryption and tell me which is better for cloud computing.
Status: ANSWERED  |  complexity: simple  |  75.1s

PLAN:
  1. [compare] Compare symmetric and asymmetric encryption focusing on suitability for cloud computing

EXECUTION:
  1. compare {"item_a": "symmetric encryption", "item_b": "asymmetric encryption", "aspect": "suitability for cloud computing"}
       thou

# Part 7: Standard RAG vs Agentic RAG
Same questions, two systems. **Standard RAG** = retrieve top-5 and answer. **Agentic RAG** = plan, tools, verification, multi-step.
Read the answers and score them (completeness, correctness, sources); the table shows tools used, sources and time.

In [24]:
# Baseline: Standard RAG = retrieve top-k, answer directly (no planning, no grading, no rewrite, no tools)
def standard_rag(question: str, k: int = 5) -> dict:
    t0 = time.time()
    is_ar = bool(ARABIC_RE.search(question))
    hits = retrieve(question, k=k)
    ctx = [{"text": h["text"], "meta": h["meta"], "grade": h["score"]} for h in hits]
    ans = call_llm(ANSWER_SYSTEM, f"Question: {question}\nAnswer language: {'Arabic' if is_ar else 'English'}\n\nContext:\n{build_context(ctx)}", max_tokens=2000).strip()
    if "NOT_ENOUGH_INFO" in ans:
        return {"question": question, "status": "not_found", "answer": scope_reply(question, is_ar),
                "sources": [], "seconds": round(time.time() - t0, 1)}
    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", ans) if 1 <= int(n) <= len(ctx)}) or list(range(1, len(ctx) + 1))
    sources = [{"id": i, "type": "doc", "label": f"{ctx[i-1]['meta']['course']} | {ctx[i-1]['meta']['source']} | {ctx[i-1]['meta']['loc']}"} for i in cited]
    return {"question": question, "status": "answered", "answer": ans, "sources": sources,
            "seconds": round(time.time() - t0, 1)}

# Complex questions where Standard RAG should struggle (edit them to match your documents)
complex_questions = [
    "Compare symmetric and asymmetric encryption and tell me which is better for cloud computing.",
    "If a cloud server costs 0.12 dollars per hour, how much do 3 servers cost for 30 days running 24/7, and what is cloud computing?",
    "Summarize the main topics of M269 and explain how they differ from TM112.",
    "How many chunks does each course have in the knowledge base?",
    "Which class has the highest number of instances?",
    "Which classes have the lowest scores across the YOLO models?",
]

rows = []
for q in complex_questions:
    s = standard_rag(q);  time.sleep(1)
    a = run_agent(q);     time.sleep(1)
    print("=" * 90 + f"\nQ: {q}\n")
    print(f"--- STANDARD RAG ({s['seconds']}s, {len(s['sources'])} sources) ---\n{s['answer']}\n")
    print(f"--- AGENTIC RAG ({a['seconds']}s, {len(a['sources'])} sources, tools: {[x['action'] for x in a['steps'] if x['action'] != 'final_answer']}) ---\n{a['answer']}\n")
    rows.append({"question": q[:60] + "...",
                 "standard_status": s["status"], "standard_sources": len(s["sources"]), "standard_sec": s["seconds"],
                 "agentic_status": a["status"], "agentic_sources": len(a["sources"]), "agentic_sec": a["seconds"],
                 "agent_tools": ", ".join(sorted({x["action"] for x in a["steps"] if x["action"] != "final_answer"}))})
pd.DataFrame(rows)

Q: Compare symmetric and asymmetric encryption and tell me which is better for cloud computing.

--- STANDARD RAG (2.4s, 4 sources) ---
Based on the provided context, here is the comparison between symmetric and asymmetric encryption and their suitability for cloud computing:

**Comparison**
*   **Speed and Performance:** Symmetric encryption is significantly faster than asymmetric encryption. Modern CPUs can perform symmetric encryption in hardware, and it can run at high speeds even on modest computers [2][4]. In contrast, asymmetric cryptography is computationally expensive and time-consuming [3].
*   **Key Size and Management:** Symmetric encryption uses small keys, making key generation and usage relatively quick [2]. Asymmetric encryption uses much larger keys, which are slower to create and use [2].
*   **Data Volume:** Symmetric encryption is well-suited for encrypting any amount of data, including streams where the final size is unknown (e.g., internet telephone calls) [1]. As

,question,standard_status,standard_sources,standard_sec,agentic_status,agentic_sources,agentic_sec,agent_tools
0,Compare symmetric and asymmetric encryption an...,answered,4,2.4,answered,5,134.7,"compare, search_kb"
1,"If a cloud server costs 0.12 dollars per hour,...",answered,2,25.9,answered,4,60.1,"calculator, search_kb"
2,Summarize the main topics of M269 and explain ...,answered,4,1.2,answered,6,112.7,"compare, summarize"
3,How many chunks does each course have in the k...,not_found,0,32.6,answered,1,15.1,analyze_data
4,Which class has the highest number of instance...,not_found,0,1.1,answered,1,2.2,analyze_data
5,Which classes have the lowest scores across th...,not_found,0,0.4,answered,1,61.8,analyze_data


# Part 8: Gradio app
Question box, course filter, **mode switch (Agentic / Standard)**, answer, sources, plan, and the tool-execution trace.
`share=True` gives a public link while the Colab session is running.

In [ ]:
import gradio as gr

def _fmt_sources(sources):
    if not sources:
        return "_No sources_"
    return "\n".join(f"- **[{s['id']}]** {s['label']}" for s in sources)

def _fmt_plan(res):
    plan = res.get("plan")
    if not plan:
        return "_Standard RAG has no planning step._"
    if not plan["plan"]:
        return f"Complexity: **{plan['complexity']}**. No tools needed."
    lines = [f"Complexity: **{plan['complexity']}**"]
    lines += [f"{s['step']}. `{s['tool']}`: {s['goal']}" for s in plan["plan"]]
    return "\n".join(lines)

def _fmt_trace(res):
    steps = res.get("steps")
    if not steps:
        return "_No tool calls._"
    out = []
    for i, s in enumerate(steps, 1):
        out.append(f"**Step {i}: `{s['action']}`**  \n_{s['thought']}_")
        if s["input"]:
            out.append(f"Input: `{json.dumps(s['input'], ensure_ascii=False)}`")
        if s["observation"]:
            out.append("Result: " + s["observation"][:300].replace("\n", " ") + " ...")
    return "\n\n".join(out)

def ui_ask(question, course_choice, mode, max_steps):
    if not question or not question.strip():
        return "Please type a question.", "", "", ""
    course = None if course_choice == "All courses" else course_choice
    try:
        res = run_agent(question, course=course, max_steps=int(max_steps)) if mode == "Agentic RAG" \
              else standard_rag(question)
    except Exception as e:
        return f"⚠️ Error: {e.__class__.__name__}: {str(e)[:300]}", "", "", ""
    return res["answer"], _fmt_plan(res), _fmt_trace(res), _fmt_sources(res["sources"])

EXAMPLES = [
    ["What is symmetric encryption?", "All courses", "Agentic RAG", 6],
    ["Compare symmetric and asymmetric encryption and tell me which is better for cloud computing.", "All courses", "Agentic RAG", 6],
    ["If a cloud server costs 0.12 dollars per hour, how much do 3 servers cost for 30 days, and what is cloud computing?", "All courses", "Agentic RAG", 6],
    ["How many chunks does each course have in the knowledge base?", "All courses", "Agentic RAG", 6],
    ["Which class has the highest number of instances?", "All courses", "Agentic RAG", 6],
    ["Which YOLO model has the highest average score across all classes, and how much is 10% of that average?", "All courses", "Agentic RAG", 6],
    ["ما هو التشفير المتماثل؟", "All courses", "Agentic RAG", 6],
]

with gr.Blocks(title="Agentic RAG Course Assistant") as demo:
    gr.Markdown("# 🎓 Agentic RAG: Course Knowledge Assistant\n"
                "The agent **plans**, picks **tools** (search, summarize, compare, data analysis, calculator), "
                "verifies what it retrieves, and answers **with sources**.")
    with gr.Row():
        with gr.Column(scale=2):
            q = gr.Textbox(label="Your question (English or Arabic)", lines=3)
            with gr.Row():
                course_dd = gr.Dropdown(["All courses"] + COURSES, value="All courses", label="Course filter")
                mode = gr.Radio(["Agentic RAG", "Standard RAG"], value="Agentic RAG", label="Mode")
                steps_sl = gr.Slider(2, 8, value=6, step=1, label="Max agent steps")
            btn = gr.Button("Ask", variant="primary")
            gr.Examples(EXAMPLES, inputs=[q, course_dd, mode, steps_sl])
        with gr.Column(scale=3):
            ans = gr.Markdown(label="Answer")
            with gr.Accordion("Sources", open=True):
                src_md = gr.Markdown()
            with gr.Accordion("Agent plan", open=False):
                plan_md = gr.Markdown()
            with gr.Accordion("Execution trace (tools used)", open=False):
                trace_md = gr.Markdown()
    btn.click(ui_ask, [q, course_dd, mode, steps_sl], [ans, plan_md, trace_md, src_md])
    q.submit(ui_ask, [q, course_dd, mode, steps_sl], [ans, plan_md, trace_md, src_md])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4e9e3be40272d73b67.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Deploying permanently (Hugging Face Spaces)
1. Copy the code cells (Parts 1-8, without the `!pip` line and without the test cells) into `app.py`.
2. Build the index once, then commit the `chroma_db/` folder (or rebuild it on startup from your documents).
3. `requirements.txt`: `pypdf python-docx python-pptx chromadb sentence-transformers langchain-text-splitters pandas numpy gradio groq`
4. In the Space settings, add `GROQ_API_KEY` as a **Secret** (never put the key in the code).
5. Create a new Space with the **Gradio** SDK and push the files.